# Analysis Tree Construction

This notebook demonstrates different ways to build analysis trees: nested splits, split_by vs summarize_by vs analyze_by, and expression types.

In [1]:
import pandas as pd
import numpy as np
from pyMyriad import AnalysisTree

In [2]:
# Create sample data
np.random.seed(42)
df = pd.DataFrame(
    {
        "Gender": np.random.choice(["M", "F"], 200),
        "Country": np.random.choice(["US", "UK", "Canada"], 200),
        "Age": np.random.randint(18, 70, 200),
        "Income": np.random.normal(50000, 15000, 200),
    }
)

## Nested Splits

Chain multiple splits to create hierarchical subgroups.

In [ ]:
# Split by Gender, then by Country
atree = (
    AnalysisTree()
    .split_by("df.Gender")
    .split_by("df.Country")
    .analyze_by(mean_income=lambda df: np.mean(df.Income), count=lambda df: len(df))
)

print(atree)
print("\n" + "=" * 60 + "\n")

result = atree.run(df)
print(result)

## Conditional Splits

Create custom split levels with conditional expressions.

In [3]:
# Split by age groups using conditional expressions
atree = (
    AnalysisTree()
    .split_by(
        label="Age Group",
        young=lambda df: df.Age < 40,
        middle=lambda df: (df.Age >= 40) & (df.Age < 60),
        senior=lambda df: df.Age >= 60,
    )
    .analyze_by(mean_income=lambda df: np.mean(df.Income), count=lambda df: len(df))
)

print(atree)
print("\n" + "=" * 60 + "\n")

result = atree.run(df)
print(result)

Analysis Tree
└─ Split Node Age Group: [young: young=lambda df: df.Age < 40, -- middle: middle=lambda df: (df.Age >= 40) & (df.Age < 60), -- senior: senior=lambda df: df.Age >= 60]
   └─ Analysis Node: 
      ├─ mean_income: mean_income=lambda df: np.mean(df.Income),
      └─ count: count=lambda df: len(df)



Data Tree
└─ Split: Age Group
   ├─ young
   │  └─ analysis: 
   │     ├─ mean_income: 50403.4332341076
   │     └─ count: 71
   ├─ middle
   │  └─ analysis: 
   │     ├─ mean_income: 51775.248733897315
   │     └─ count: 86
   └─ senior
      └─ analysis: 
         ├─ mean_income: 49688.39217948439
         └─ count: 43



## analyze_by vs summarize_by

- **analyze_by**: Terminal node - cannot add more splits after it
- **summarize_by**: Non-terminal node - can continue splitting after it

In [4]:
# Use summarize_by to compute statistics and continue splitting
atree = (
    AnalysisTree()
    .split_by("df.Gender")
    .summarize_by(mean_income=lambda df: np.mean(df.Income))
    .split_by("df.Country")
    .analyze_by(mean_income=lambda df: np.mean(df.Income), count=lambda df: len(df))
)

print(atree)
print("\n" + "=" * 60 + "\n")

result = atree.run(df)
print(result)

Analysis Tree
└─ Split Node df.Gender: [df.Gender]
   ├─ Analysis Node: 
   │  └─ mean_income: <function>
   └─ Split Node df.Country: [df.Country]
      └─ Analysis Node: 
         ├─ mean_income: mean_income=lambda df: np.mean(df.Income),
         └─ count: count=lambda df: len(df)



Data Tree
└─ Split: df.Gender
   ├─ F
   │  ├─ analysis: 
   │  │  └─ mean_income: 52707.24730553199
   │  └─ Split: df.Country
   │     ├─ Canada
   │     │  └─ analysis: 
   │     │     ├─ mean_income: 53512.89128192307
   │     │     └─ count: 31
   │     ├─ UK
   │     │  └─ analysis: 
   │     │     ├─ mean_income: 51707.300070992904
   │     │     └─ count: 37
   │     └─ US
   │        └─ analysis: 
   │           ├─ mean_income: 53082.96869333898
   │           └─ count: 32
   └─ M
      ├─ analysis: 
      │  └─ mean_income: 48971.91283901439
      └─ Split: df.Country
         ├─ Canada
         │  └─ analysis: 
         │     ├─ mean_income: 49774.613175279046
         │     └─ count: 35
    

## String Expressions vs Lambda Functions

Both approaches are equivalent - choose based on your preference.

In [5]:
# Using string expressions
atree_str = (
    AnalysisTree()
    .split_by("df.Gender")
    .analyze_by(mean_income="np.mean(df.Income)", max_age="np.max(df.Age)")
)

# Using lambda functions
atree_lambda = (
    AnalysisTree()
    .split_by("df.Gender")
    .analyze_by(
        mean_income=lambda df: np.mean(df.Income), max_age=lambda df: np.max(df.Age)
    )
)

print("String expression results:")
print(atree_str.run(df))
print("\nLambda function results:")
print(atree_lambda.run(df))

String expression results:
Data Tree
└─ Split: df.Gender
   ├─ F
   │  └─ analysis: 
   │     ├─ mean_income: 52707.24730553199
   │     └─ max_age: 69
   └─ M
      └─ analysis: 
         ├─ mean_income: 48971.91283901439
         └─ max_age: 69


Lambda function results:
Data Tree
└─ Split: df.Gender
   ├─ F
   │  └─ analysis: 
   │     ├─ mean_income: 52707.24730553199
   │     └─ max_age: 69
   └─ M
      └─ analysis: 
         ├─ mean_income: 48971.91283901439
         └─ max_age: 69



## Complex Analysis with Multiple Levels

Combine everything for sophisticated multi-level analyses.

In [6]:
atree = (
    AnalysisTree()
    .split_by("df.Gender")
    .summarize_by(overall_mean="np.mean(df.Income)", overall_count="len(df)")
    .split_by(
        label="Age Group", young=lambda df: df.Age < 40, older=lambda df: df.Age >= 40
    )
    .split_by("df.Country")
    .analyze_by(
        mean_income=lambda df: np.mean(df.Income),
        std_income=lambda df: np.std(df.Income),
        count=lambda df: len(df),
    )
)

print(atree)
print("\n" + "=" * 60 + "\n")

result = atree.run(df)
print(result)

Analysis Tree
└─ Split Node df.Gender: [df.Gender]
   ├─ Analysis Node: 
   │  ├─ overall_mean: np.mean(df.Income)
   │  └─ overall_count: len(df)
   └─ Split Node Age Group: [young: young=lambda df: df.Age < 40, -- older: older=lambda df: df.Age >= 40]
      └─ Split Node df.Country: [df.Country]
         └─ Analysis Node: 
            ├─ mean_income: mean_income=lambda df: np.mean(df.Income),
            ├─ std_income: std_income=lambda df: np.std(df.Income),
            └─ count: count=lambda df: len(df)



Data Tree
└─ Split: df.Gender
   ├─ F
   │  ├─ analysis: 
   │  │  ├─ overall_mean: 52707.24730553199
   │  │  └─ overall_count: 100
   │  └─ Split: Age Group
   │     ├─ young
   │     │  └─ Split: df.Country
   │     │     ├─ Canada
   │     │     │  └─ analysis: 
   │     │     │     ├─ mean_income: 57234.346777694904
   │     │     │     ├─ std_income: 15189.01661869367
   │     │     │     └─ count: 16
   │     │     ├─ UK
   │     │     │  └─ analysis: 
   │     │     │    

## Ghost Levels and Empty Groups

### What is a ghost level?

A **ghost level** is a split group that is defined but contains zero observations.
It can arise in two situations:

1. **`pd.Categorical` columns** — a category is part of the dtype's definition but absent from the current slice of data.
2. **Keyword-expression (`kwexpr`) splits** — a condition is specified explicitly but never satisfied by any row in the current subset.

---

### Case 1 — `pd.Categorical` ghost levels (handled automatically)

When the split expression evaluates to a `pd.Categorical` series, `pandas.groupby` would historically return a group for *every category in the dtype*, even unpopulated ones. pyMyriad calls `groupby(..., observed=True)` internally, so only categories that actually appear in the data are returned.

No action needed from you — this is the default behaviour.

In [10]:
# "East" and "West" are defined in the dtype but absent from the data.
# Without observed=True they would produce phantom empty groups.
df_cat = pd.DataFrame(
    {
        "Region": pd.Categorical(
            np.random.choice(["North", "South"], 200),
            categories=["North", "South", "East", "West"],
        ),
        "Income": np.random.normal(50000, 15000, 200),
    }
)

atree_cat = (
    AnalysisTree()
    .split_by("df.Region")
    .analyze_by(
        mean_income=lambda df: np.mean(df.Income),
        count=lambda df: len(df),
    )
)

result_cat = atree_cat.run(df_cat)
print(result_cat)

Data Tree
└─ Split: df.Region
   ├─ North
   │  └─ analysis: 
   │     ├─ mean_income: 50189.624445799556
   │     └─ count: 97
   └─ South
      └─ analysis: 
         ├─ mean_income: 48887.04934945851
         └─ count: 103



### Case 2 — Keyword-expression ghost levels (opt-in control with `drop_empty`)

When you define split groups explicitly with keyword expressions, some conditions may not be satisfied by every subset of the data. By default (`drop_empty=False`) those empty levels are **kept** in the result — useful when you want a consistent structure across subgroups regardless of whether each level is populated.

Set `drop_empty=True` to silently **discard** levels that produce zero rows.

In [11]:
# Restrict to low-income observations — nobody qualifies for the "high" group.
df_low = df[df.Income < 40000].copy()

# --- drop_empty=False (default): empty "high" level is preserved ---
atree_keep = (
    AnalysisTree()
    .split_by(
        label="Income Group",
        low=lambda df: df.Income < 40000,
        high=lambda df: df.Income >= 40000,
        drop_empty=False,
    )
    .analyze_by(count=lambda df: len(df))
)

result_keep = atree_keep.run(df_low)
print(result_keep)

print()

# --- drop_empty=True: the empty "high" level is discarded ---
atree_drop = (
    AnalysisTree()
    .split_by(
        label="Income Group",
        low=lambda df: df.Income < 40000,
        high=lambda df: df.Income >= 40000,
        drop_empty=True,
    )
    .analyze_by(count=lambda df: len(df))
)

result_drop = atree_drop.run(df_low)
print(result_drop)

Data Tree
└─ Split: Income Group
   ├─ low
   │  └─ analysis: 
   │     └─ count: 42
   └─ high
      └─ analysis: 
         └─ count: 0


Data Tree
└─ Split: Income Group
   └─ low
      └─ analysis: 
         └─ count: 42

